In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio



In [2]:
datapath = "./creditcardfraud/creditcard.csv"
data = pd.read_csv(datapath)


In [9]:
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

# Load the dataset
df = pd.read_csv(datapath)

# Select columns to discretize
cols_to_bin = ['V1', 'V2', 'V3', 'Amount']

# Initialize KBinsDiscretizer
kbd = KBinsDiscretizer(n_bins=12, encode='ordinal', strategy='quantile')

# Fit and transform selected columns
binned_data = kbd.fit_transform(df[cols_to_bin])

# Create new column names
binned_cols = [f'{col}_bin' for col in cols_to_bin]

# Add binned columns to the dataframe
df[binned_cols] = binned_data

# (Optional) Preview the result
print(df[cols_to_bin + binned_cols].head())
df
# (Optional) Save to new CSV
#df.to_csv("creditcard_binned.csv", index=False)


         V1        V2        V3  Amount  V1_bin  V2_bin  V3_bin  Amount_bin
0 -1.359807 -0.072781  2.536347  149.62     1.0     5.0    11.0        10.0
1  1.191857  0.266151  0.166480    2.69     8.0     7.0     5.0         2.0
2 -1.358354 -1.340163  1.773209  378.66     1.0     1.0    10.0        11.0
3 -0.966272 -0.185226  1.792993  123.50     2.0     4.0    11.0         9.0
4 -1.158233  0.877737  1.548718   69.99     2.0     9.0    10.0         8.0


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,Class,V1_bin,V2_bin,V3_bin,Amount_bin
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.128539,-0.189115,0.133558,-0.021053,149.62,0,1.0,5.0,11.0,10.0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0.167170,0.125895,-0.008983,0.014724,2.69,0,8.0,7.0,5.0,2.0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0,1.0,1.0,10.0,11.0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0.647376,-0.221929,0.062723,0.061458,123.50,0,2.0,4.0,11.0,9.0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.206010,0.502292,0.219422,0.215153,69.99,0,2.0,9.0,10.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,1.436807,0.250034,0.943651,0.823731,0.77,0,0.0,11.0,0.0,0.0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,-0.606624,-0.395255,0.068472,-0.053527,24.79,0,3.0,5.0,11.0,6.0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.265745,-0.087371,0.004455,-0.026561,67.88,0,10.0,4.0,0.0,8.0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,-0.569159,0.546668,0.108821,0.104533,10.00,0,5.0,8.0,7.0,4.0


In [12]:
df.sample(frac=0.6).to_csv("./creditcardfraud/creditcard_bin.csv", index=False)


In [23]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

# Attempt to import cuDF for GPU operations.
try:
    import cudf
    gpu_available = True
except ImportError:
    gpu_available = False

class MultiFeatureCategoricalFraudTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, category_cols, label_col='is_fraud', top_k=10, fraud_coverage=0.8):
        """
        Parameters:
        - category_cols: list of categorical feature names.
        - label_col: name of the binary fraud column (1 indicates fraud).
        - top_k: maximum number of top categories to one-hot encode.
        - fraud_coverage: cumulative fraud fraction threshold for selecting top categories.
        """
        self.category_cols = category_cols
        self.label_col = label_col
        self.top_k = top_k
        self.fraud_coverage = fraud_coverage
        self.gpu_available = gpu_available

        self.top_categories_map_ = {}
        self.label_encoding_maps_ = {}
        self.inverse_label_maps_ = {}
        self.best_cuts_ = {}

    def _process_single_feature(self, dfx, feature):
        # Compute fraud statistics per category.
        fraud_df = dfx[dfx[self.label_col] == 1]
        fraud_counts = fraud_df.groupby(feature).agg({self.label_col: "count"}).rename(columns={self.label_col: "fraud_count"})
        total_counts = dfx.groupby(feature).agg({feature: "count"}).rename(columns={feature: "total"})
        stats = fraud_counts.join(total_counts, how="outer").fillna(0)
        stats['fraud_rate'] = stats['fraud_count'] / stats['total']
        stats['fraud_score'] = np.sqrt(stats['fraud_rate']) * stats['fraud_count']
        stats = stats.sort_values('fraud_score', ascending=False)

        # Determine top risky categories.
        total_fraud = stats['fraud_count'].sum()
        cumulative = 0
        top_cats = []
        for cat, row in stats.iterrows():
            if len(top_cats) >= self.top_k:
                break
            cumulative += row['fraud_count']
            top_cats.append(cat)
            if cumulative / total_fraud >= self.fraud_coverage:
                break

        self.top_categories_map_[feature] = top_cats
        self.best_cuts_[feature] = stats.loc[top_cats[-1], 'fraud_score'] if top_cats else None

        # Create label encoding map for the remaining categories (ordered ascending by risk).
        remaining_stats = stats[~stats.index.isin(top_cats)].sort_values('fraud_score', ascending=True)
        label_map = {cat: i + 1 for i, cat in enumerate(remaining_stats.index.tolist())}
        self.label_encoding_maps_[feature] = label_map
        self.inverse_label_maps_[feature] = {v: k for k, v in label_map.items()}

        # Create a temporary column for one-hot encoding:
        dfx[f'__{feature}_temp__'] = dfx[feature].apply(lambda x: x if x in top_cats else "OTHER")

        # Create the label encoded column for categories not in top_cats.
        dfx[f'{feature}_label_enc'] = dfx[feature].apply(lambda x: 0 if x in top_cats else label_map.get(x, 0))
        
        # One-hot encode the top categories.
        if self.gpu_available:
            one_hot = cudf.get_dummies(dfx[f'__{feature}_temp__'], prefix=f"{feature}_cat")
        else:
            one_hot = pd.get_dummies(dfx[f'__{feature}_temp__'], prefix=f"{feature}_cat")
        
        # Remove the temporary column and join the one-hot encoded features.
        dfx = dfx.drop(columns=[f'__{feature}_temp__'])
        dfx = dfx.join(one_hot)
        return dfx

    def fit(self, X, y=None):
        # We'll use the provided DataFrame directly.
        if self.gpu_available:
            dfx = cudf.DataFrame.from_pandas(X) if not isinstance(X, cudf.DataFrame) else X.copy()
        else:
            dfx = X.copy()

        # Process each feature to build the mappings.
        for feature in self.category_cols:
            # We don't modify dfx here since fit only needs to compute the mappings.
            fraud_df = dfx[dfx[self.label_col] == 1]
            fraud_counts = fraud_df.groupby(feature).agg({self.label_col: "count"}).rename(columns={self.label_col: "fraud_count"})
            total_counts = dfx.groupby(feature).agg({feature: "count"}).rename(columns={feature: "total"})
            stats = fraud_counts.join(total_counts, how="outer").fillna(0)
            stats['fraud_rate'] = stats['fraud_count'] / stats['total']
            stats['fraud_score'] = np.sqrt(stats['fraud_rate']) * stats['fraud_count']
            stats = stats.sort_values('fraud_score', ascending=False)
    
            total_fraud = stats['fraud_count'].sum()
            cumulative = 0
            top_cats = []
            for cat, row in stats.iterrows():
                if len(top_cats) >= self.top_k:
                    break
                cumulative += row['fraud_count']
                top_cats.append(cat)
                if cumulative / total_fraud >= self.fraud_coverage:
                    break

            self.top_categories_map_[feature] = top_cats
            self.best_cuts_[feature] = stats.loc[top_cats[-1], 'fraud_score'] if top_cats else None

            remaining_stats = stats[~stats.index.isin(top_cats)].sort_values('fraud_score', ascending=True)
            label_map = {cat: i + 1 for i, cat in enumerate(remaining_stats.index.tolist())}
            self.label_encoding_maps_[feature] = label_map
            self.inverse_label_maps_[feature] = {v: k for k, v in label_map.items()}
        return self

    def transform(self, X):
        if self.gpu_available:
            dfx = cudf.DataFrame.from_pandas(X) if not isinstance(X, cudf.DataFrame) else X.copy()
        else:
            dfx = X.copy()

        for feature in self.category_cols:
            top_cats = self.top_categories_map_.get(feature, [])

            # One-hot encode top categories only (others will be ignored in one-hot)
            dfx[f'__{feature}_temp__'] = dfx[feature]
            if self.gpu_available:
                one_hot = cudf.get_dummies(dfx[f'__{feature}_temp__'], prefix=f"{feature}_cat", dtype="int8")
            else:
                one_hot = pd.get_dummies(dfx[f'__{feature}_temp__'], prefix=f"{feature}_cat", dtype="int8")

            # Keep only one-hot columns for top categories
            valid_one_hot_cols = [f"{feature}_cat_{cat}" for cat in top_cats if f"{feature}_cat_{cat}" in one_hot.columns]
            one_hot = one_hot[valid_one_hot_cols]

            # Label encode all values (no "OTHER" fallback)
            label_map = self.label_encoding_maps_.get(feature, {})
            dfx[f'{feature}_label_enc'] = dfx[feature].apply(lambda x: label_map.get(x, 0))

            # Drop temp column and join one-hot
            dfx = dfx.drop(columns=[f'__{feature}_temp__'])
            dfx = dfx.join(one_hot)

        return dfx.to_pandas() if self.gpu_available else dfx


    def inverse_label(self, feature, value):
        """Convert a label encoded value back to its original category for a given feature."""
        return self.inverse_label_maps_.get(feature, {}).get(value, "Unknown")
    
    def get_top_categories(self, feature):
        """Retrieve the list of top categories for a given feature."""
        return self.top_categories_map_.get(feature, [])
    
    def get_best_cut(self, feature):
        """Retrieve the best cut (risk score threshold) for the top categories of a given feature."""
        return self.best_cuts_.get(feature, None)

# Example usage:
if __name__ == "__main__":
    # Create a sample DataFrame.
    datapath = "./creditcardfraud/creditcard_bin.csv"
    category_cols = ['V1_bin', 'V2_bin', 'V3_bin', 'Amount_bin']
    label_col = 'Class'
    df = pd.read_csv(datapath)

    transformer = MultiFeatureCategoricalFraudTransformer(
        category_cols=category_cols,
        label_col=label_col,
        top_k=10,
        fraud_coverage=0.6
    )
    
    # Fit the transformer (builds mappings) and transform the data.
    df_transformed = transformer.fit_transform(df)
    
    print("Transformed DataFrame:")
    print(df_transformed.head())
    
    # Retrieve some details:
    print("\nBest cut for merchant_category:", transformer.get_best_cut('merchant_category'))
    print("Top risky device types:", transformer.get_top_categories('device_type'))
    print("Original device_type for label 1:", transformer.inverse_label('device_type', 1))


Transformed DataFrame:
       Time        V1        V2        V3        V4        V5        V6  \
0    1718.0 -1.518956  1.095081  0.612097 -0.041425  0.761113  0.055685   
1   62877.0  1.108417 -1.204981  1.162947 -0.317237 -2.026645 -0.694744   
2  146871.0  1.996553 -0.381120 -0.670863  0.089211 -0.115310  0.347652   
3   34636.0 -0.755436  1.285099  1.652601  1.520321  0.700615  0.170646   
4  150866.0 -1.975217  2.115231 -1.724706 -1.004831  0.192180 -1.174983   

         V7        V8        V9  ...  V1_bin_cat_6.0  V2_bin_label_enc  \
0  0.486362  0.079485 -0.143197  ...               0                11   
1 -1.073366  0.046245  0.074705  ...               0                 4   
2 -0.639954  0.206396  1.086198  ...               0                 7   
3  0.698103  0.257911 -1.004875  ...               0                11   
4  0.348190  0.676270  0.597598  ...               0                 0   

   V2_bin_cat_11.0  V3_bin_label_enc  V3_bin_cat_0.0  Amount_bin_label_enc  \
0  

In [20]:
transformer.label_encoding_maps_


{'V1_bin': {11.0: 1,
  10.0: 2,
  9.0: 3,
  5.0: 4,
  4.0: 5,
  3.0: 6,
  8.0: 7,
  7.0: 8,
  2.0: 9,
  1.0: 10,
  6.0: 11},
 'V2_bin': {2.0: 1,
  4.0: 2,
  6.0: 3,
  1.0: 4,
  7.0: 5,
  9.0: 6,
  3.0: 7,
  5.0: 8,
  0.0: 9,
  8.0: 10,
  10.0: 11},
 'V3_bin': {10.0: 1,
  8.0: 2,
  11.0: 3,
  7.0: 4,
  5.0: 5,
  6.0: 6,
  9.0: 7,
  4.0: 8,
  3.0: 9,
  2.0: 10,
  1.0: 11},
 'Amount_bin': {4.0: 1,
  8.0: 2,
  5.0: 3,
  7.0: 4,
  6.0: 5,
  10.0: 6,
  2.0: 7,
  3.0: 8,
  9.0: 9}}

In [21]:
transformer.label_encoding_maps_

{'V1_bin': {11.0: 1,
  10.0: 2,
  9.0: 3,
  5.0: 4,
  4.0: 5,
  3.0: 6,
  8.0: 7,
  7.0: 8,
  2.0: 9,
  1.0: 10,
  6.0: 11},
 'V2_bin': {2.0: 1,
  4.0: 2,
  6.0: 3,
  1.0: 4,
  7.0: 5,
  9.0: 6,
  3.0: 7,
  5.0: 8,
  0.0: 9,
  8.0: 10,
  10.0: 11},
 'V3_bin': {10.0: 1,
  8.0: 2,
  11.0: 3,
  7.0: 4,
  5.0: 5,
  6.0: 6,
  9.0: 7,
  4.0: 8,
  3.0: 9,
  2.0: 10,
  1.0: 11},
 'Amount_bin': {4.0: 1,
  8.0: 2,
  5.0: 3,
  7.0: 4,
  6.0: 5,
  10.0: 6,
  2.0: 7,
  3.0: 8,
  9.0: 9}}